# 📈 CAPM Factor Model Analysis

This notebook demonstrates the Capital Asset Pricing Model (CAPM) regression analysis for a portfolio of stocks.

**CAPM Formula:**
$$R_i - R_f = \alpha_i + \beta_i (R_m - R_f) + \epsilon_i$$

Where:
- $\alpha$ (Alpha) = Abnormal return / Jensen's Alpha
- $\beta$ (Beta) = Systematic risk / Market sensitivity
- $R_f$ = Risk-free rate
- $R_m$ = Market return (S&P 500)

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

# Import our factor model package
from factor_model import (
    get_stock_returns,
    get_market_returns,
    get_risk_free_rate,
    calculate_excess_returns,
    merge_data,
    single_factor_regression,
    run_all_regressions,
)

# Configure plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

## 1. Data Collection

Download historical stock and market data from Yahoo Finance.

In [ ]:
# Configuration
SYMBOLS = ["AAPL", "MSFT", "GOOGL", "JPM", "XOM"]
START_DATE = "2023-01-01"
END_DATE = "2026-01-01"

# Download data
print("📥 Downloading stock data...")
stock_returns = get_stock_returns(SYMBOLS, START_DATE, END_DATE)

print("📥 Downloading market data (S&P 500)...")
market_returns = get_market_returns(START_DATE, END_DATE)

# Calculate excess returns
rf_rate = get_risk_free_rate()
print(f"\n📊 Daily risk-free rate: {rf_rate:.6f} ({rf_rate * 252:.2%} annualized)")

stock_excess = calculate_excess_returns(stock_returns, rf_rate)
market_excess = calculate_excess_returns(market_returns, rf_rate)

# Merge data
data = merge_data(stock_excess, market_excess)
print(f"\n✅ Data shape: {data.shape[0]} trading days × {data.shape[1]} assets")

In [ ]:
# Preview the data
data.head(10)

## 2. Exploratory Data Analysis

In [ ]:
# Summary statistics
summary = pd.DataFrame({
    'Mean (Daily)': data.mean(),
    'Std (Daily)': data.std(),
    'Mean (Annual)': data.mean() * 252,
    'Std (Annual)': data.std() * np.sqrt(252),
    'Sharpe Ratio': (data.mean() * 252) / (data.std() * np.sqrt(252)),
    'Min': data.min(),
    'Max': data.max()
})
summary.round(4)

In [ ]:
# Correlation matrix
fig, ax = plt.subplots(figsize=(10, 8))
correlation = data.corr()
mask = np.triu(np.ones_like(correlation, dtype=bool))
sns.heatmap(
    correlation, 
    mask=mask,
    annot=True, 
    fmt='.2f', 
    cmap='RdYlGn',
    center=0,
    square=True,
    linewidths=0.5,
    ax=ax
)
plt.title('Correlation Matrix: Excess Returns', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cumulative returns plot
cumulative = (1 + data).cumprod()

fig, ax = plt.subplots(figsize=(12, 6))
for col in cumulative.columns:
    ax.plot(cumulative.index, cumulative[col], label=col, linewidth=1.5)

ax.axhline(y=1, color='black', linestyle='--', alpha=0.5)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Cumulative Return', fontsize=12)
ax.set_title('Cumulative Excess Returns', fontsize=14, fontweight='bold')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig('plots/cumulative_returns.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. CAPM Regression Analysis

In [ ]:
# Run CAPM regressions for all stocks
market = data["Market"]
stocks = data.drop(columns=["Market"])

print("🔬 Running CAPM Regressions...\n")
results_df = run_all_regressions(stocks, market)

In [ ]:
# Format results table
results_display = results_df.copy()
results_display['Alpha (Annual)'] = results_display['Alpha'] * 252
results_display['Alpha Sig.'] = results_display['Alpha_pvalue'].apply(
    lambda x: '✅' if x < 0.05 else '❌'
)
results_display['Beta Sig.'] = results_display['Beta_pvalue'].apply(
    lambda x: '✅' if x < 0.05 else '❌'
)

display_cols = ['Stock', 'Alpha (Annual)', 'Alpha Sig.', 'Beta', 'Beta Sig.', 'R_squared']
results_display[display_cols].round(4)

## 4. Visualizations

In [ ]:
# Beta comparison chart
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#2ecc71' if b < 1 else '#e74c3c' for b in results_df['Beta']]
bars = ax.bar(results_df['Stock'], results_df['Beta'], color=colors, edgecolor='black', linewidth=1.2)

ax.axhline(y=1, color='black', linestyle='--', linewidth=2, label='Market (β=1)')

for bar, beta in zip(bars, results_df['Beta']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03, 
            f'{beta:.2f}', ha='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Stock', fontsize=12)
ax.set_ylabel('Beta (β)', fontsize=12)
ax.set_title('CAPM Beta Comparison\n(Green = Defensive, Red = Aggressive)', fontsize=14, fontweight='bold')
ax.legend()
ax.set_ylim(0, max(results_df['Beta']) + 0.3)

plt.tight_layout()
plt.savefig('plots/beta_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Individual regression plots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, stock in enumerate(stocks.columns):
    ax = axes[i]
    
    # Scatter plot
    ax.scatter(market, stocks[stock], alpha=0.4, s=20)
    
    # Regression line
    model = single_factor_regression(stocks[stock], market)
    x_line = np.linspace(market.min(), market.max(), 100)
    y_line = model.params.iloc[0] + model.params.iloc[1] * x_line
    ax.plot(x_line, y_line, color='red', linewidth=2, 
            label=f'β={model.params.iloc[1]:.2f}, R²={model.rsquared:.2f}')
    
    ax.set_xlabel('Market Excess Return')
    ax.set_ylabel(f'{stock} Excess Return')
    ax.set_title(f'{stock}', fontsize=12, fontweight='bold')
    ax.legend(loc='upper left', fontsize=9)
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)

# Hide empty subplot
axes[-1].axis('off')

plt.suptitle('CAPM Regression: Stock vs Market Returns', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plots/regression_grid.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Risk-Return scatter plot
fig, ax = plt.subplots(figsize=(10, 8))

annual_returns = data.mean() * 252
annual_vol = data.std() * np.sqrt(252)

for stock in data.columns:
    ax.scatter(annual_vol[stock], annual_returns[stock], s=150, alpha=0.8)
    ax.annotate(stock, (annual_vol[stock], annual_returns[stock]), 
                xytext=(5, 5), textcoords='offset points', fontsize=11, fontweight='bold')

ax.set_xlabel('Annualized Volatility (Risk)', fontsize=12)
ax.set_ylabel('Annualized Excess Return', fontsize=12)
ax.set_title('Risk-Return Profile', fontsize=14, fontweight='bold')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('plots/risk_return.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Key Insights

### Beta Interpretation
| Stock | Beta | Classification |
|-------|------|----------------|
| β > 1 | Aggressive | More volatile than market |
| β = 1 | Neutral | Moves with market |
| β < 1 | Defensive | Less volatile than market |

### Alpha Interpretation
- **Positive Alpha**: Stock outperforms on risk-adjusted basis
- **Negative Alpha**: Stock underperforms on risk-adjusted basis
- **Statistical Significance**: p-value < 0.05 indicates reliable alpha

In [ ]:
# Final summary
print("="*60)
print("📊 CAPM ANALYSIS SUMMARY")
print("="*60)

highest_beta = results_df.loc[results_df['Beta'].idxmax()]
lowest_beta = results_df.loc[results_df['Beta'].idxmin()]
highest_r2 = results_df.loc[results_df['R_squared'].idxmax()]

print(f"\n🔺 Most Aggressive (Highest β): {highest_beta['Stock']} (β = {highest_beta['Beta']:.2f})")
print(f"🔻 Most Defensive (Lowest β): {lowest_beta['Stock']} (β = {lowest_beta['Beta']:.2f})")
print(f"📈 Best Market Fit (Highest R²): {highest_r2['Stock']} (R² = {highest_r2['R_squared']:.2f})")

sig_alphas = results_df[results_df['Alpha_pvalue'] < 0.05]
if len(sig_alphas) > 0:
    print(f"\n✅ Stocks with significant alpha: {', '.join(sig_alphas['Stock'].values)}")
else:
    print(f"\n❌ No stocks show statistically significant alpha (p < 0.05)")
    print("   This is consistent with efficient market hypothesis!")

In [ ]:
# Save results to data/ directory
results_df.to_csv('../data/capm_results.csv', index=False)
print("✅ Results saved to data/capm_results.csv")